# Video-ai Google Colab GPU worker

In **Runtime → Change runtime type**, select an available GPU. Colab may assign CPU or end a free session; this worker reports not-ready unless both NVIDIA and ONNX Runtime CUDA are detected. Add `VIDEO_AI_WORKER_TOKEN` in the Colab Secrets panel and grant this notebook access. Use the same random token as `COLAB_WORKER_TOKEN` in the Video-ai Codespace.

Run setup after each fresh session. To connect, expose only port 8001 through an HTTPS tunnel you configure yourself. The worker binds to localhost; never display or share the token.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('/content/Video-ai')
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/cymmetry1/Video-ai.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only', 'origin', 'main'], check=True)
DLC_REV = '0bae1a549014d86457548b05a01a3f7682cdda65'
DLC = ROOT / 'Deep-Live-Cam'
if not (DLC / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/hacksider/Deep-Live-Cam.git', str(DLC)], check=True)
subprocess.run(['git', '-C', str(DLC), 'fetch', '--depth', '1', 'origin', DLC_REV], check=True)
subprocess.run(['git', '-C', str(DLC), 'checkout', '--detach', DLC_REV], check=True)
os.chdir(ROOT)
print('Pinned Video-ai and Deep-Live-Cam source is ready.')

In [ ]:
%pip install -r requirements.txt -r .devcontainer/requirements-gpu-worker.txt

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime'], check=False, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', 'onnxruntime-gpu==1.26.0'], check=True)
print('Verified GPU wheel installation path; CUDA provider will be checked next.')

In [ ]:
from google.colab import userdata
import onnxruntime as ort
import os
import subprocess
import sys

token = userdata.get('VIDEO_AI_WORKER_TOKEN')
if not token:
    raise RuntimeError('Add VIDEO_AI_WORKER_TOKEN in the Colab Secrets panel and grant notebook access.')
os.environ.update({
    'VIDEO_AI_WORKER_TOKEN': token,
    'VIDEO_AI_WORKER_ONLY': '1',
    'DLC_ENGINE': str(ROOT / 'Deep-Live-Cam'),
    'DLC_PYTHON': sys.executable,
    'EXECUTION_PROVIDER': 'cuda',
    'VIDEO_AI_EXECUTION_PROVIDER': 'cuda',
    'QT_QPA_PLATFORM': 'offscreen',
    'U2NET_HOME': str(ROOT / 'work' / 'models' / 'rembg'),
})
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True)
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('ONNX Runtime CUDA provider is unavailable; this session is not GPU-ready.')
print(gpu.stdout.strip())
print('ONNX Runtime CUDA is available.')

In [ ]:
subprocess.run([sys.executable, '.devcontainer/setup_models.py'], check=True)
subprocess.run([sys.executable, '.devcontainer/setup_quality_models.py', '--quality', 'balanced', '--quality', 'best'], check=True)
print('GPU model installation and CUDA inference checks passed.')

In [ ]:
import time
import urllib.request

worker = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8001'],
    cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
for _ in range(60):
    if worker.poll() is not None:
        raise RuntimeError('GPU worker server exited during startup.')
    try:
        request = urllib.request.Request('http://127.0.0.1:8001/api/worker/health', headers={'Authorization': f'Bearer {token}'})
        with urllib.request.urlopen(request, timeout=5) as response:
            print(response.read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('GPU worker server did not start.')
print('Worker is bound to localhost:8001. Configure your permitted HTTPS tunnel separately.')